In [0]:
-- CREATE OR REPLACE TEMPORARY VIEW reference_file_v1 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Load base reference file
   --------------------------------------------------------------------------- */
base_table AS (
    SELECT *
    FROM cmpa_insights_internal_schema.reference_file_12_12_2025
),

/* ---------------------------------------------------------------------------
   STEP 1: Split records that already have an HCO NPI
   --------------------------------------------------------------------------- */
vod_search_base_table_v1 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 2: Records without HCO NPI but with HCP NPI
   --------------------------------------------------------------------------- */
vod_search_base_table_v2 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi = '-'
      AND hcp_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 3: Fetch HCP VID from VOD using HCP NPI
   --------------------------------------------------------------------------- */
hcp_vid AS (
    SELECT
        a.*,
        b.vid__v AS hcp_vid
    FROM vod_search_base_table_v2 a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* ---------------------------------------------------------------------------
   STEP 4: Pull active HCP–HCO affiliations from VOD and rank by recency
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v1 AS (
    SELECT
        a.hcp_npi,
        c.vid__v AS hco_vid,
        c.corporate_name__v AS hco_name,
        c.npi_num__v AS hco_npi,
        b.modified_date__v,
        b.status_update_time__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

/* ---------------------------------------------------------------------------
   STEP 5: Select most recent active affiliation per HCP
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v2 AS (
    SELECT
        hcp_npi,
        hco_npi,
        hco_name
    FROM ranked_vod_affiliations_v1
    WHERE rn = 1
      AND hco_npi IS NOT NULL
),

/* ---------------------------------------------------------------------------
   STEP 6: Apply VOD-derived HCOs to missing records
   --------------------------------------------------------------------------- */
vod_affiliations_implementation AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(b.hco_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM vod_search_base_table_v2 a
    LEFT JOIN ranked_vod_affiliations_v2 b
        ON a.hcp_npi = b.hcp_npi
),

/* ---------------------------------------------------------------------------
   STEP 7: Union records with original and VOD-derived affiliations
   --------------------------------------------------------------------------- */
union_after_vod_check AS (
    SELECT * FROM vod_search_base_table_v1
    UNION
    SELECT * FROM vod_affiliations_implementation
),

/* ---------------------------------------------------------------------------
   STEP 8: Split records with and without HCO affiliations
   --------------------------------------------------------------------------- */
hcp_with_no_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi = '-'
),

hcp_with_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 9: Backfill missing HCOs using Komodo affiliations
   --------------------------------------------------------------------------- */
pulling_affiliations_from_komodo AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_primary_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(c.organization_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM hcp_with_no_affiliations a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.com_raw.kom_providers c
        ON b.hco_primary_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 10: Combine all affiliations (VOD + Komodo)
   --------------------------------------------------------------------------- */
finalizing_affiliations AS (
    SELECT * FROM hcp_with_affiliations
    UNION
    SELECT * FROM pulling_affiliations_from_komodo
),

/* ---------------------------------------------------------------------------
   STEP 11: Manual NPI remapping (Jess file logic)
   --------------------------------------------------------------------------- */
npi_mapping AS (
    SELECT current_npi, current_name, mapped_npi, mapped_name
    FROM (
        VALUES
        ('1144211301','Atrium Health Wake Forest Baptist Medical Center','1295789907','Atrium Health'),
        ('1184779332','Childrens Healthcare Of Atlanta Scottish Rite Hospital','1235339227','Emory University Hospital'),
        ('1851458038','Dr Patrick Leavey MD Office','1235582925','UT Health'),
        ('1225259039','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1831318856','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1356496772','Kaiser Permanente Fontana Medical Center','1013062769','Kaiser Permanente San Diego Medical Center'),
        ('1003947599','Univ. Pediatric Associates Inc.','1144266024','Indiana University Health'),
        ('1205822236','Yale Medicine','1013924182','Yale-New Haven Hospital')
    ) t (current_npi, current_name, mapped_npi, mapped_name)
),

/* ---------------------------------------------------------------------------
   STEP 12: Apply manual NPI and name overrides
   --------------------------------------------------------------------------- */
jess_file_implementation AS (
    SELECT
        * EXCEPT (a.hco_npi, a.hco_name),
        COALESCE(b.mapped_npi, a.hco_npi) AS hco_npi,
        COALESCE(b.mapped_name, a.hco_name) AS hco_name
    FROM finalizing_affiliations a
    LEFT JOIN npi_mapping b
        ON a.hco_npi = b.current_npi
),

/* ---------------------------------------------------------------------------
   STEP 13: Set HCO target flag based on approved target NPIs
   --------------------------------------------------------------------------- */
updating_hco_target_flag AS (
    SELECT
        a.* EXCEPT (hco_target),
        CASE
            WHEN (a.hco_npi IN ( /* predefined target list */ 
                '1932280666','1912939703','1891765178','1861439952','1851458038',
                '1831318856','1760480503','1760476659','1750482022','1750458485',
                '1700128592','1689747552','1679973364','1669683512','1669462420',
                '1669429577','1659877280','1649347469','1649261462','1639370059',
                '1609824010','1598784555','1578693321','1568596765','1548212988',
                '1477643690','1477549756','1467525790','1447423959','1437365186',
                '1396882205','1376544320','1366556227','1366515488','1356496772',
                '1346297843','1336495910','1336245828','1326092404','1295789907',
                '1285832634','1285647933','1285174649','1275694184','1275564098',
                '1265694442','1235582925','1235339227','1235234535','1235214834',
                '1235148594','1225259039','1225249865','1215921457','1205935012',
                '1205822236','1194787218','1184779332','1184649345','1164686879',
                '1164426896','1154302727','1144548322','1144266024','1144211301',
                '1114969169','1114924834','1104819366','1093894131','1093808040',
                '1083949382','1083789630','1083630073','1073053757','1063702785',
                '1053632463','1043447253','1033439732','1023188851','1023105400',
                '1013924372','1013924182','1013143213','1013062769','1003961251',
                '1003947599','1003878539','1003102781','1003063280'
            ) or a.hco_npi in (select distinct primary_npi from com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi where secondary_npi in ('1932280666','1912939703','1891765178','1861439952','1851458038',
                '1831318856','1760480503','1760476659','1750482022','1750458485',
                '1700128592','1689747552','1679973364','1669683512','1669462420',
                '1669429577','1659877280','1649347469','1649261462','1639370059',
                '1609824010','1598784555','1578693321','1568596765','1548212988',
                '1477643690','1477549756','1467525790','1447423959','1437365186',
                '1396882205','1376544320','1366556227','1366515488','1356496772',
                '1346297843','1336495910','1336245828','1326092404','1295789907',
                '1285832634','1285647933','1285174649','1275694184','1275564098',
                '1265694442','1235582925','1235339227','1235234535','1235214834',
                '1235148594','1225259039','1225249865','1215921457','1205935012',
                '1205822236','1194787218','1184779332','1184649345','1164686879',
                '1164426896','1154302727','1144548322','1144266024','1144211301',
                '1114969169','1114924834','1104819366','1093894131','1093808040',
                '1083949382','1083789630','1083630073','1073053757','1063702785',
                '1053632463','1043447253','1033439732','1023188851','1023105400',
                '1013924372','1013924182','1013143213','1013062769','1003961251',
                '1003947599','1003878539','1003102781','1003063280')))
            THEN a.hco_npi
            ELSE '-'
        END AS hco_target
    FROM jess_file_implementation a
)
select * 
from updating_hco_target_flag
where hcp_npi in ('1013003094')

In [0]:
/* ============================================================================
   PURPOSE
   ----------------------------------------------------------------------------
   This script builds a final HCP–HCO reference view by:
   1. Starting from a base reference file
   2. Retaining records that already have HCO NPIs
   3. Deriving missing HCO affiliations using VOD (Salesforce) data
   4. Backfilling remaining missing affiliations using Komodo
   5. Applying manual NPI remapping (Jess file logic)
   6. Flagging target HCOs
   7. Mapping secondary NPIs to primary NPIs (Julie file logic)
   8. Enriching HCO ZIPs using VOD and Komodo
   9. Reassigning territory & region based on ZIP
  10. Producing a final, clean HCP–HCO reference output
   ============================================================================ */

CREATE OR REPLACE TEMPORARY VIEW reference_file_v1 AS
WITH

/* ---------------------------------------------------------------------------
   STEP 0: Load base reference file
   --------------------------------------------------------------------------- */
base_table AS (
    SELECT *
    FROM cmpa_insights_internal_schema.reference_file_12_12_2025
),

/* ---------------------------------------------------------------------------
   STEP 1: Split records that already have an HCO NPI
   --------------------------------------------------------------------------- */
vod_search_base_table_v1 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 2: Records without HCO NPI but with HCP NPI
   --------------------------------------------------------------------------- */
vod_search_base_table_v2 AS (
    SELECT *
    FROM base_table
    WHERE hco_npi = '-'
      AND hcp_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 3: Fetch HCP VID from VOD using HCP NPI
   --------------------------------------------------------------------------- */
hcp_vid AS (
    SELECT
        a.*,
        b.vid__v AS hcp_vid
    FROM vod_search_base_table_v2 a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* ---------------------------------------------------------------------------
   STEP 4: Pull active HCP–HCO affiliations from VOD and rank by recency
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v1 AS (
    SELECT
        a.hcp_npi,
        c.vid__v AS hco_vid,
        c.corporate_name__v AS hco_name,
        c.npi_num__v AS hco_npi,
        b.modified_date__v,
        b.status_update_time__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

/* ---------------------------------------------------------------------------
   STEP 5: Select most recent active affiliation per HCP
   --------------------------------------------------------------------------- */
ranked_vod_affiliations_v2 AS (
    SELECT
        hcp_npi,
        hco_npi,
        hco_name
    FROM ranked_vod_affiliations_v1
    WHERE rn = 1
      AND hco_npi IS NOT NULL
),

/* ---------------------------------------------------------------------------
   STEP 6: Apply VOD-derived HCOs to missing records
   --------------------------------------------------------------------------- */
vod_affiliations_implementation AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(b.hco_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM vod_search_base_table_v2 a
    LEFT JOIN ranked_vod_affiliations_v2 b
        ON a.hcp_npi = b.hcp_npi
),

/* ---------------------------------------------------------------------------
   STEP 7: Union records with original and VOD-derived affiliations
   --------------------------------------------------------------------------- */
union_after_vod_check AS (
    SELECT * FROM vod_search_base_table_v1
    UNION
    SELECT * FROM vod_affiliations_implementation
),

/* ---------------------------------------------------------------------------
   STEP 8: Split records with and without HCO affiliations
   --------------------------------------------------------------------------- */
hcp_with_no_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi = '-'
),

hcp_with_affiliations AS (
    SELECT *
    FROM union_after_vod_check
    WHERE hco_npi != '-'
),

/* ---------------------------------------------------------------------------
   STEP 9: Backfill missing HCOs using Komodo affiliations
   --------------------------------------------------------------------------- */
pulling_affiliations_from_komodo AS (
    SELECT
        a.hcp_npi,
        a.hcp_target,
        a.hcp_first_name,
        a.hcp_last_name,
        a.hcp_zipcode,
        COALESCE(b.hco_primary_npi, '-') AS hco_npi,
        a.hco_target,
        COALESCE(c.organization_name, '-') AS hco_name,
        a.hco_zip,
        a.territory,
        a.region
    FROM hcp_with_no_affiliations a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.com_raw.kom_providers c
        ON b.hco_primary_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 10: Combine all affiliations (VOD + Komodo)
   --------------------------------------------------------------------------- */
finalizing_affiliations AS (
    SELECT * FROM hcp_with_affiliations
    UNION
    SELECT * FROM pulling_affiliations_from_komodo
),

/* ---------------------------------------------------------------------------
   STEP 11: Manual NPI remapping (Jess file logic)
   --------------------------------------------------------------------------- */
npi_mapping AS (
    SELECT current_npi, current_name, mapped_npi, mapped_name
    FROM (
        VALUES
        ('1144211301','Atrium Health Wake Forest Baptist Medical Center','1295789907','Atrium Health'),
        ('1184779332','Childrens Healthcare Of Atlanta Scottish Rite Hospital','1235339227','Emory University Hospital'),
        ('1851458038','Dr Patrick Leavey MD Office','1235582925','UT Health'),
        ('1225259039','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1831318856','Greenwood Genetics Center Inc.','1649261462','Greenwood Genetics Center Inc.'),
        ('1356496772','Kaiser Permanente Fontana Medical Center','1013062769','Kaiser Permanente San Diego Medical Center'),
        ('1003947599','Univ. Pediatric Associates Inc.','1144266024','Indiana University Health'),
        ('1205822236','Yale Medicine','1013924182','Yale-New Haven Hospital')
    ) t (current_npi, current_name, mapped_npi, mapped_name)
),

/* ---------------------------------------------------------------------------
   STEP 12: Apply manual NPI and name overrides
   --------------------------------------------------------------------------- */
jess_file_implementation AS (
    SELECT
        * EXCEPT (a.hco_npi, a.hco_name),
        COALESCE(b.mapped_npi, a.hco_npi) AS hco_npi,
        COALESCE(b.mapped_name, a.hco_name) AS hco_name
    FROM finalizing_affiliations a
    LEFT JOIN npi_mapping b
        ON a.hco_npi = b.current_npi
),

/* ---------------------------------------------------------------------------
   STEP 13: Set HCO target flag based on approved target NPIs
   --------------------------------------------------------------------------- */
-- updating_hco_target_flag AS (
--     SELECT
--         a.* EXCEPT (hco_target),
--         CASE
--             WHEN a.hco_npi IN ( /* predefined target list */ 
--                 '1932280666','1912939703','1891765178','1861439952','1851458038',
--                 '1831318856','1760480503','1760476659','1750482022','1750458485',
--                 '1700128592','1689747552','1679973364','1669683512','1669462420',
--                 '1669429577','1659877280','1649347469','1649261462','1639370059',
--                 '1609824010','1598784555','1578693321','1568596765','1548212988',
--                 '1477643690','1477549756','1467525790','1447423959','1437365186',
--                 '1396882205','1376544320','1366556227','1366515488','1356496772',
--                 '1346297843','1336495910','1336245828','1326092404','1295789907',
--                 '1285832634','1285647933','1285174649','1275694184','1275564098',
--                 '1265694442','1235582925','1235339227','1235234535','1235214834',
--                 '1235148594','1225259039','1225249865','1215921457','1205935012',
--                 '1205822236','1194787218','1184779332','1184649345','1164686879',
--                 '1164426896','1154302727','1144548322','1144266024','1144211301',
--                 '1114969169','1114924834','1104819366','1093894131','1093808040',
--                 '1083949382','1083789630','1083630073','1073053757','1063702785',
--                 '1053632463','1043447253','1033439732','1023188851','1023105400',
--                 '1013924372','1013924182','1013143213','1013062769','1003961251',
--                 '1003947599','1003878539','1003102781','1003063280'
--             )
--             THEN a.hco_npi
--             ELSE '-'
--         END AS hco_target
--     FROM jess_file_implementation a
-- ),

updating_hco_target_flag AS (
    SELECT
        a.* EXCEPT (hco_target),
        CASE
            WHEN (a.hco_npi IN ( /* predefined target list */ 
                '1932280666','1912939703','1891765178','1861439952','1851458038',
                '1831318856','1760480503','1760476659','1750482022','1750458485',
                '1700128592','1689747552','1679973364','1669683512','1669462420',
                '1669429577','1659877280','1649347469','1649261462','1639370059',
                '1609824010','1598784555','1578693321','1568596765','1548212988',
                '1477643690','1477549756','1467525790','1447423959','1437365186',
                '1396882205','1376544320','1366556227','1366515488','1356496772',
                '1346297843','1336495910','1336245828','1326092404','1295789907',
                '1285832634','1285647933','1285174649','1275694184','1275564098',
                '1265694442','1235582925','1235339227','1235234535','1235214834',
                '1235148594','1225259039','1225249865','1215921457','1205935012',
                '1205822236','1194787218','1184779332','1184649345','1164686879',
                '1164426896','1154302727','1144548322','1144266024','1144211301',
                '1114969169','1114924834','1104819366','1093894131','1093808040',
                '1083949382','1083789630','1083630073','1073053757','1063702785',
                '1053632463','1043447253','1033439732','1023188851','1023105400',
                '1013924372','1013924182','1013143213','1013062769','1003961251',
                '1003947599','1003878539','1003102781','1003063280'
            ) or a.hco_npi in (select distinct primary_npi from com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi where secondary_npi in ('1932280666','1912939703','1891765178','1861439952','1851458038',
                '1831318856','1760480503','1760476659','1750482022','1750458485',
                '1700128592','1689747552','1679973364','1669683512','1669462420',
                '1669429577','1659877280','1649347469','1649261462','1639370059',
                '1609824010','1598784555','1578693321','1568596765','1548212988',
                '1477643690','1477549756','1467525790','1447423959','1437365186',
                '1396882205','1376544320','1366556227','1366515488','1356496772',
                '1346297843','1336495910','1336245828','1326092404','1295789907',
                '1285832634','1285647933','1285174649','1275694184','1275564098',
                '1265694442','1235582925','1235339227','1235234535','1235214834',
                '1235148594','1225259039','1225249865','1215921457','1205935012',
                '1205822236','1194787218','1184779332','1184649345','1164686879',
                '1164426896','1154302727','1144548322','1144266024','1144211301',
                '1114969169','1114924834','1104819366','1093894131','1093808040',
                '1083949382','1083789630','1083630073','1073053757','1063702785',
                '1053632463','1043447253','1033439732','1023188851','1023105400',
                '1013924372','1013924182','1013143213','1013062769','1003961251',
                '1003947599','1003878539','1003102781','1003063280')))
            THEN a.hco_npi
            ELSE '-'
        END AS hco_target
    FROM jess_file_implementation a
),

/* ---------------------------------------------------------------------------
   STEP 14: Map secondary NPIs to primary NPIs (Julie file logic)
   --------------------------------------------------------------------------- */
julie_file_mapping AS (
    SELECT
        a.*,
        COALESCE(b.primary_npi, a.hco_npi) AS hco_npi_active,
        COALESCE(c.primary_npi, a.hco_target) AS hco_target_active,
        CASE WHEN b.secondary_npi IS NOT NULL THEN 1 ELSE 0 END
            AS hco_npi_present_julies_file_flag,
        CASE
            WHEN b.primary_npi IS NOT NULL THEN b.primary_name
            ELSE a.hco_name
        END AS hco_name_active
    FROM updating_hco_target_flag a
    LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi b
        ON a.hco_npi = b.secondary_npi
    LEFT JOIN cmpa_insights_internal_schema.secondary_to_primary_npi c
        ON a.hco_target = c.secondary_npi
),

/* ---------------------------------------------------------------------------
   STEP 15: Fetch latest HCO ZIP from VOD
   --------------------------------------------------------------------------- */
hco_zip_v1 AS (
    SELECT DISTINCT
        a.npi_num__v AS hco_npi_active,
        b.postal_code_cda__v AS hco_postal_code,
        b.modified_date__v,
        ROW_NUMBER() OVER (
            PARTITION BY a.npi_num__v
            ORDER BY b.modified_date__v DESC
        ) AS rn
    FROM com_raw.vod_hco a
    JOIN com_raw.vod_address b
        ON b.entity_vid__v = a.vid__v
       AND b.entity_type__v = 'HCO'
       AND b.record_state__v = 'VALID'
       AND b.address_status__v IN ('A','DS')
       AND b.address_verification_status__v NOT IN ('NS','U')
    WHERE a.npi_num__v IN (
        SELECT DISTINCT hco_npi_active
        FROM julie_file_mapping
        WHERE hco_npi_active != '-'
    )
),

/* ---------------------------------------------------------------------------
   STEP 16: Select latest ZIP per HCO
   --------------------------------------------------------------------------- */
hco_zip_v2 AS (
    SELECT
        hco_npi_active,
        hco_postal_code
    FROM hco_zip_v1
    WHERE rn = 1
),

/* ---------------------------------------------------------------------------
   STEP 17: Backfill missing HCO ZIP using Komodo
   --------------------------------------------------------------------------- */
pulling_hco_zip_using_vod_komodo AS (
    SELECT
        a.* EXCEPT (hco_zip),
        COALESCE(b.hco_postal_code, c.provider_zip, '-') AS hco_zip
    FROM julie_file_mapping a
    LEFT JOIN hco_zip_v2 b
        ON a.hco_npi_active = b.hco_npi_active
    LEFT JOIN com_raw.kom_providers c
        ON a.hco_npi_active = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ---------------------------------------------------------------------------
   STEP 18: Reassign territory & region using ZIP
   --------------------------------------------------------------------------- */
territory_region_reassignment AS (
    SELECT
        a.* EXCEPT (territory, region),
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region
    FROM pulling_hco_zip_using_vod_komodo a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON COALESCE(
               TRY_CAST(NULLIF(a.hco_zip, '-') AS BIGINT),
               TRY_CAST(NULLIF(a.hcp_zipcode, '-') AS BIGINT)
           ) = b.zipcode
),

/* ---------------------------------------------------------------------------
   STEP 19: Final structural cleanup
   --------------------------------------------------------------------------- */
final_output_v1 AS (
    SELECT DISTINCT
        hcp_npi,
        hcp_target,
        hcp_first_name,
        hcp_last_name,
        hcp_zipcode AS hcp_zip,
        hco_npi,
        hco_npi_active,
        hco_npi_present_julies_file_flag,
        hco_target,
        hco_target_active,
        hco_name,
        hco_name_active,
        hco_zip,
        territory_id,
        territory,
        region_id,
        region
    FROM territory_region_reassignment
),

/* ---------------------------------------------------------------------------
   STEP 20: Final enrichment with HCP specialty and formatting
   --------------------------------------------------------------------------- */
final_output_v2 AS (
    SELECT DISTINCT
        hcp_npi,
        hcp_target,
        hcp_first_name,
        hcp_last_name,
        CONCAT(hcp_first_name, ' ', hcp_last_name) AS hcp_name,
        COALESCE(primary_specialty, '-') AS hcp_specialty,
        COALESCE(secondary_specialty, '-') AS hcp_secondary_specialty,
        COALESCE(hcp_zip, '-') AS hcp_zip,
        hco_npi_active AS hco_npi,
        hco_npi_present_julies_file_flag,
        hco_target_active AS hco_target,
        hco_name_active AS hco_name,
        hco_zip,
        COALESCE(CAST(TRY_CAST(territory_id AS BIGINT) AS STRING), '-') AS territory_id,
        COALESCE(territory, '-') AS territory,
        COALESCE(CAST(TRY_CAST(region_id AS BIGINT) AS STRING), '-') AS region_id,
        COALESCE(region, '-') AS region
    FROM final_output_v1
    LEFT JOIN com_raw.kom_providers
        ON hcp_npi = npi
       AND provider_type = 'INDIVIDUAL'
)

SELECT *
FROM final_output_v2;


In [0]:
select distinct PRIMARY_SPECIALTY
from com_edp_prd.com_raw.kom_providers
where PRIMARY_SPECIALTY is not null and PROVIDER_TYPE = 'INDIVIDUAL'

In [0]:
select distinct SECONDARY_SPECIALTY
from com_edp_prd.com_raw.kom_providers
where SECONDARY_SPECIALTY is not null and PROVIDER_TYPE = 'INDIVIDUAL'

In [0]:
select distinct npi
from (select distinct NPI, PRIMARY_SPECIALTY, SECONDARY_SPECIALTY
from com_raw.kom_providers
where PROVIDER_TYPE = 'INDIVIDUAL')
where NOT (
    -- Primary specialty is in the exclusion list (or is NULL)
    (
        PRIMARY_SPECIALTY IN (
            'Anesthesiologist Assistant',
            'Anesthesiology',
            'Dentist',
            'Dietitian, Registered',
            'Emergency Medical Technician, Basic',
            'Emergency Medicine',
            'General Acute Care Hospital',
            'Nurse Anesthetist, Certified Registered',
            'Obstetrics & Gynecology',
            'Pathology',
            'Radiology',
            'Urology'
        )
        OR PRIMARY_SPECIALTY IS NULL
    )
    -- AND secondary specialty is NOT in the exception list
    AND (
        SECONDARY_SPECIALTY NOT IN (
            -- Behavioral/Mental Healthcare
            'Child & Adolescent Psychiatry',
            'Psychiatry',
            -- Pediatric Medicine
            'Adolescent Medicine',
            'Developmental - Behavioral Pediatrics',
            'Neonatal-Perinatal Medicine',
            'Nutrition, Pediatric',
            'Oncology, Pediatrics',
            'Pediatric Cardiology',
            'Pediatric Critical Care Medicine',
            'Pediatric Dermatology',
            'Pediatric Emergency Medicine',
            'Pediatric Endocrinology',
            'Pediatric Gastroenterology',
            'Pediatric Hematology-Oncology',
            'Pediatric Infectious Diseases',
            'Pediatric Nephrology',
            'Pediatric Ophthalmology and Strabismus Specialist',
            'Pediatric Orthopaedic Surgery',
            'Pediatric Otolaryngology',
            'Pediatric Pulmonology',
            'Pediatric Radiology',
            'Pediatric Rehabilitation Medicine',
            'Pediatric Rheumatology',
            'Pediatric Surgery',
            'Pediatrics',
            -- Medical Genetics
            'Clinical Biochemical Genetics',
            'Clinical Genetics (M.D.)',
            'Clinical Molecular Genetics',
            'Ph.D. Medical Genetics',
            -- Neurology
            'Neurodevelopmental Disabilities',
            'Neurology',
            'Neurology with Special Qualifications in Child Neurology',
            'Neuroradiology'
        )
        OR SECONDARY_SPECIALTY IS NULL
    )
)

In [0]:
with all_claims as (
  SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Dx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Dx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761','E763')
  AND TRANSACTION_STATUS = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events 
WHERE NDC11 IN ('54092070001','540920700')
UNION
SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE, 'Tx' as claim_type, coalesce(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) as plan_id
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001','540920700')
  AND TRANSACTION_RESULT = 'PAID'
UNION
SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE, 'Tx' as claim_type, KH_PLAN_ID as plan_id
FROM com_edp_prd.com_raw.kom_medical_events 
WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                          '38206','38230','38232','38240','38241','38242','38243','38250')
),
relevant_patients as (
  select *
  from all_claims
  where patient_id in (select distinct patient_id from com_edp_prd.cmpa_insights_internal_schema.patient360)
),
patient_geography AS (
  SELECT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),
pulling_relevant_info as (
  select a.patient_id, a.npi, a.fill_date, b.PATIENT_YOB, (year(current_date) - year(b.PATIENT_YOB)) as age, c.PATIENT_STATE, d.FIRST_NAME, d.LAST_NAME, concat(d.FIRST_NAME, ' ', d.LAST_NAME) as hcp_name, d.PRIMARY_SPECIALTY, d.SECONDARY_SPECIALTY, a.claim_type, e.PAYER_NAME, e.INSURANCE_GROUP
  from relevant_patients as a
  left join com_edp_prd.com_raw.kom_patient_demographics as b on a.patient_id = b.PATIENT_ID
  left join patient_geography as c on a.patient_id = c.PATIENT_ID
  left join com_edp_prd.com_raw.kom_providers as d on a.npi = d.npi and d.provider_type = 'INDIVIDUAL'
  left join com_edp_prd.com_raw.kom_plans as e on a.plan_id = e.KH_PLAN_ID
)
select * from pulling_relevant_info